# VisionQ AI — Simple Multimodal Demo

This notebook sends an image and a text question to a vision-capable Groq model.

**What you'll learn**
- Load your API key from `.env`
- Convert an image to Base64
- Send image + question to a multimodal LLM
- Read the model's answer

This first version keeps things simple. It does not use LangChain or an agent yet.

In [1]:
import os
import base64
import mimetypes

from dotenv import load_dotenv
from groq import Groq

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("Add GROQ_API_KEY to your .env file.")

client = Groq(api_key=api_key)

MODEL = "qwen/qwen3.8-27b"
print("Groq client is ready.")

Groq client is ready.


## 3. Choose an image

Put an image (PNG, JPG, or WebP) in your project folder, then update the path below.

For example, use a photo of a question paper or a page from your notes.

In [3]:
image_path = "sample_paper.png"  # Change this to your image filename

if not os.path.isfile(image_path):
    raise FileNotFoundError(f"Image not found: {image_path}")

mime_type, _ = mimetypes.guess_type(image_path)

if mime_type not in ("image/jpeg", "image/png", "image/webp"):
    raise ValueError("Please use a JPG, PNG, or WebP image.")

print("Image found:", image_path)
print("Image type:", mime_type)

Image found: sample_paper.png
Image type: image/png


## 4. Convert the image to Base64

Base64 converts the image bytes into text so they can be included in the API request. **Base64 does not understand the image**; the vision model does.

In [4]:
with open(image_path, "rb") as image_file:
    image_bytes = image_file.read()

base64_image = base64.b64encode(image_bytes).decode("utf-8")
image_url = f"data:{mime_type};base64,{base64_image}"

print("Image encoded successfully.")

Image encoded successfully.


## 5. Ask a question about the image

Change `question` to tell the model what you want it to do.

In [5]:
question = "Read the questions in this image and answer them clearly."

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": question},
                {
                    "type": "image_url",
                    "image_url": {"url": image_url}
                }
            ]
        }
    ],
    temperature=0.2,
    max_completion_tokens=1500,
)

answer = response.choices[0].message.content
print(answer)

Based on the image provided, here are the clear answers to the questions:

**Given:**
- $\triangle ABC$ is a right-angled triangle with $\angle B = 90^\circ$.
- $AB = 6 \text{ cm}$
- $BC = 8 \text{ cm}$
- $AC$ is the hypotenuse.

---

**(a) Find the length of the hypotenuse $AC$.**

Using the Pythagorean theorem:
$$
AC^2 = AB^2 + BC^2
$$
$$
AC^2 = 6^2 + 8^2 = 36 + 64 = 100
$$
$$
AC = \sqrt{100} = 10 \text{ cm}
$$

✅ **Answer: $AC = 10 \text{ cm}$**

---

**(b) Find the area of $\triangle ABC$.**

Area of a right-angled triangle = $\frac{1}{2} \times \text{base} \times \text{height}$

Take $BC = 8 \text{ cm}$ as base and $AB = 6 \text{ cm}$ as height (since they are perpendicular).

$$
\text{Area} = \frac{1}{2} \times 8 \times 6 = 24 \text{ cm}^2
$$

✅ **Answer: $24 \text{ cm}^2$**

---

**(c) Find the value of $\sin C$, $\cos C$ and $\tan C$.**

In right triangle $ABC$, with respect to angle $C$:

- Opposite side to $\angle C$ is $AB = 6 \text{ cm}$
- Adjacent side to $\angle C$ is $BC

## 6. Try different prompts

Run the next cell to change the task. You can ask the model to explain a diagram, read handwritten text, or solve a question from the image.

In [6]:
question = "Explain the main concept shown in this image in simple language."

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": question},
                {
                    "type": "image_url",
                    "image_url": {"url": image_url}
                }
            ]
        }
    ],
    temperature=0.2,
    max_completion_tokens=1500,
)

print(response.choices[0].message.content)

The image shows a **Class 10 Mathematics problem** about a **right-angled triangle**, specifically triangle ABC, where:

- Angle B is 90° (so it’s the right angle).
- Side AB = 6 cm (one leg).
- Side BC = 8 cm (the other leg).
- Side AC is the **hypotenuse** (the longest side, opposite the right angle).

---

### 🎯 Main Concept:
This problem is designed to test your understanding of **basic properties and calculations in right-angled triangles**, including:

1. **Pythagoras’ Theorem** — to find the length of the hypotenuse.
2. **Area of a triangle** — using base and height.
3. **Trigonometric ratios** — sine, cosine, tangent for an angle.
4. **Circle geometry** — whether a point lies on a circle based on distance from center.

---

### 🔍 Let’s break down each part simply:

#### (a) Find the length of the hypotenuse AC.
→ Use **Pythagoras’ Theorem**:  
> In a right triangle, *hypotenuse² = leg₁² + leg₂²*

So:  
AC² = AB² + BC² = 6² + 8² = 36 + 64 = 100  
→ AC = √100 = **10 cm**

✅ So, t

## What this notebook demonstrates

- **Multimodal input:** text and image are sent in one message.
- **Base64 encoding:** prepares a local image for the request.
- **Vision LLM:** interprets the image and generates an answer.

**Next learning step:** after this works, add a small tool-calling example in a separate section. That will demonstrate how the model can request a Python function instead of answering entirely on its own.